# Discovering Codelists (Building a Metadata Explorer)

In the previous notebook, we explored the **Data Structure Definition (DSD)** of a BIS Dataflow and identified its **Dimensions**, **Measures**, and **Attributes**.

Several dimensions were linked to **Codelists**, but we only discovered their IDs (for example, `CL_FREQ` and `CL_AREA`).

In this notebook, we will retrieve and explore these Codelists to understand the valid values that can be used for each dimension.

By the end of this notebook, we will be able to answer questions such as:

- Which countries are available?
- Which reporting frequencies are supported?
- Which borrower sectors exist?
- Which units of measurement can be queried?

Without downloading a single observation, we will build a simple **Metadata Explorer** that helps us understand the vocabulary of a BIS dataset and construct valid SDMX queries.

In [1]:
import pandas as pd
import sdmx

In [2]:
client = sdmx.Client("BIS")

In [3]:
flows = client.dataflow()

In [4]:
flow = flows.dataflow["WS_TC"]

print(flow)

WS_TC


In [5]:
dsd_response = client.get(
    "datastructure",
    resource_id=flow.structure.id
)

In [6]:
# ----------------------------------------------------
# Explore the StructureMessage/dsd_response
# ----------------------------------------------------

collections = []

for name in dir(dsd_response):

    if name.startswith("_"):
        continue

    try:
        obj = getattr(dsd_response, name)

        # Only keep collection-like objects
        if hasattr(obj, "__len__") and not isinstance(obj, str):

            collections.append({
                "Collection": name,
                "Type": type(obj).__name__,
                "Count": len(obj)
            })

    except Exception:
        pass

df_structure = (
    pd.DataFrame(collections)
      .sort_values("Collection")
      .reset_index(drop=True)
)

df_structure

,Collection,Type,Count
0,categorisation,DictLike,0
1,category_scheme,DictLike,0
2,codelist,DictLike,14
3,concept_scheme,DictLike,1
4,constraint,DictLike,0
5,custom_type_scheme,DictLike,0
6,dataflow,DictLike,1
7,hierarchical_codelist,DictLike,0
8,hierarchy,DictLike,0
9,metadataflow,DictLike,0


In [7]:
# ----------------------------------------------------
# Explore the Codelist Collection
# ----------------------------------------------------

print(type(dsd_response.codelist))

print(f"\nTotal Codelists : {len(dsd_response.codelist)}")

print("\nCodelist IDs:\n")

for cid in dsd_response.codelist.keys():
    print(cid)

<class 'sdmx.dictlike.DictLike'>

Total Codelists : 14

Codelist IDs:

CL_ADJUST
CL_AREA
CL_BIS_UNIT
CL_COLLECTION
CL_CONF_STATUS
CL_DECIMALS
CL_FREQ
CL_OBS_STATUS
CL_SUB_CHANNEL
P
CL_TC_BORROWERS
CL_TC_LENDERS
CL_UNIT_MULT
CL_VALUATION


In [8]:
# ----------------------------------------------------
# Retrieve one Codelist
# ----------------------------------------------------

freq = dsd_response.codelist["CL_FREQ"]

print(type(freq))
print(freq)

<class 'sdmx.model.common.Codelist'>
CL_FREQ


In [9]:
# ----------------------------------------------------
# Explore the Codelist object
# ----------------------------------------------------

for attr in dir(freq):
    if not attr.startswith("_"):
        try:
            value = getattr(freq, attr)

            # Avoid printing huge collections
            if hasattr(value, "__len__") and not isinstance(value, str):
                print(f"{attr:25}: {type(value).__name__} (len={len(value)})")
            else:
                print(f"{attr:25}: {value}")

        except Exception:
            pass

annotations              : list (len=0)
append                   : <bound method ItemScheme.append of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
compare                  : <bound method Comparable.compare of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
description              : 
eval_annotation          : <bound method AnnotableArtefact.eval_annotation of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
extend                   : <bound method ItemScheme.extend of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
get                      : <bound method ItemScheme.get of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
get_annotation           : <bound method AnnotableArtefact.get_annotation of <Codelist BIS:CL_FREQ(1.0) (8 items): Code list for Frequency (FREQ)>>
get_hierarchical         : <bound method ItemScheme.get_hierarchical of <Codelist BIS:CL_FREQ(1.0) (8 ite

In [10]:
# ----------------------------------------------------
# Build a summary of all Codelists
# ----------------------------------------------------

codelists = []

for cl in dsd_response.codelist.values():

    codelists.append({
        "Codelist ID": cl.id,
        "Name": str(cl.name),
        "Maintainer": str(cl.maintainer),
        "Version": cl.version,
        "No. of Codes": len(cl.items)
    })

df_codelists = (
    pd.DataFrame(codelists)
      .sort_values("Codelist ID")
      .reset_index(drop=True)
)

df_codelists

,Codelist ID,Name,Maintainer,Version,No. of Codes
0,CL_ADJUST,Adjustment codelist,BIS,1.0,4
1,CL_AREA,Reference area code list,BIS,1.0,101
2,CL_BIS_UNIT,BIS_Unit,BIS,1.0,1096
3,CL_COLLECTION,Collection,BIS,1.0,10
4,CL_CONF_STATUS,Observation confidentiality code list,BIS,1.0,5
5,CL_DECIMALS,"Decimals codelist (BIS, ECB)",BIS,1.0,17
6,CL_FREQ,Code list for Frequency (FREQ),BIS,1.0,8
7,CL_OBS_STATUS,"Observation status codelist (BIS, ECB, Eurosta...",BIS,1.0,16
8,CL_SUB_CHANNEL,Submission channel,BIS,1.0,3
9,CL_TC_BORROWERS,Borrowers,BIS,1.0,5


In [11]:
freq.items

{'A': <Code A: Annual>,
 'B': <Code B: Daily - business week (not supported)>,
 'D': <Code D: Daily>,
 'E': <Code E: Event (not supported)>,
 'H': <Code H: Half-yearly>,
 'M': <Code M: Monthly>,
 'Q': <Code Q: Quarterly>,
 'W': <Code W: Weekly>}

We've learned three important things:
<br>
1. A StructureMessage contains the codelists required for the selected DSD.
<br>
2. A Codelist stores its values in the items dictionary.
<br>
3. Each entry in items is a Code object.
<br>
So the next logical step is to explore a Code object.

In [12]:
# ----------------------------------------------------
# Explore a Code object
# ----------------------------------------------------

annual = freq.items["A"]

print(type(annual))
print(annual)

<class 'sdmx.model.common.Code'>
A


In [13]:
# ----------------------------------------------------
# Inspect every property
# ----------------------------------------------------

for attr in dir(annual):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(annual, attr)

        if hasattr(value, "__len__") and not isinstance(value, str):
            print(f"{attr:25}: {type(value).__name__} (len={len(value)})")
        else:
            print(f"{attr:25}: {value}")

    except Exception:
        pass

annotations              : list (len=0)
append_child             : <bound method Item.append_child of <Code A: Annual>>
child                    : list (len=0)
compare                  : <bound method Comparable.compare of <Code A: Annual>>
description              : 
eval_annotation          : <bound method AnnotableArtefact.eval_annotation of <Code A: Annual>>
get_annotation           : <bound method AnnotableArtefact.get_annotation of <Code A: Annual>>
get_child                : <bound method Item.get_child of <Code A: Annual>>
get_scheme               : <bound method Item.get_scheme of <Code A: Annual>>
hierarchical_id          : A
id                       : A
name                     : Annual
parent                   : Codelist (len=8)
pop_annotation           : <bound method AnnotableArtefact.pop_annotation of <Code A: Annual>>
uri                      : None
urn                      : urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:CL_FREQ(1.0).A


In [14]:
# ----------------------------------------------------
# Build Frequency Metadata
# ----------------------------------------------------

rows = []

for code in freq.items.values():

    rows.append({
        "Code": code.id,
        "Description": str(code.name),
        "URN": code.urn
    })

df_freq = pd.DataFrame(rows)

df_freq

,Code,Description,URN
0,A,Annual,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,B,Daily - business week (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,D,Daily,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,E,Event (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,H,Half-yearly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,M,Monthly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,Q,Quarterly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,W,Weekly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [15]:
metadata = {}

for cl in dsd_response.codelist.values():

    rows = []

    for code in cl.items.values():

        rows.append({
            "Code": code.id,
            "Description": str(code.name)
        })

    metadata[cl.id] = pd.DataFrame(rows)

In [16]:
metadata["CL_FREQ"]

,Code,Description
0,A,Annual
1,B,Daily - business week (not supported)
2,D,Daily
3,E,Event (not supported)
4,H,Half-yearly
5,M,Monthly
6,Q,Quarterly
7,W,Weekly


In [17]:
metadata["CL_AREA"]

,Code,Description
0,1X,ECB
1,4T,Emerging market economies (aggregate)
2,5A,All reporting economies
3,5R,Advanced economies
4,AE,United Arab Emirates
...,...,...
96,VN,Vietnam
97,XM,Euro area
98,XW,World
99,ZA,South Africa


In [18]:
metadata["CL_TC_BORROWERS"]

,Code,Description
0,C,Non financial sector
1,G,General government
2,H,Households & NPISHs
3,N,Non-financial corporations
4,P,Private non-financial sector


In [19]:
metadata["CL_BIS_UNIT"]

,Code,Description
0,000,Unknown
1,001,100 - yield
2,002,EUR / MWh
3,003,"Index, 1996 Jan 2 = 100"
4,004,Euro / troy ounce
...,...,...
1091,ZMK,Zambian Kwacha
1092,ZMW,Zambian Kwacha
1093,ZWD,Zimbabwe Dollar
1094,ZWG,Zimbabwe Gold


In [20]:
# ----------------------------------------------------
# Build a Metadata Explorer
# ----------------------------------------------------

metadata = {}

for cl in dsd_response.codelist.values():

    rows = []

    for code in cl.items.values():

        rows.append({
            "Code": code.id,
            "Description": str(code.name),
            "URN": code.urn
        })

    metadata[cl.id] = pd.DataFrame(rows)

In [21]:
metadata.keys()

dict_keys(['CL_ADJUST', 'CL_AREA', 'CL_BIS_UNIT', 'CL_COLLECTION', 'CL_CONF_STATUS', 'CL_DECIMALS', 'CL_FREQ', 'CL_OBS_STATUS', 'CL_SUB_CHANNEL', 'P', 'CL_TC_BORROWERS', 'CL_TC_LENDERS', 'CL_UNIT_MULT', 'CL_VALUATION'])

In [22]:
metadata["CL_AREA"].head(20)

,Code,Description,URN
0,1X,ECB,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,4T,Emerging market economies (aggregate),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,5A,All reporting economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,5R,Advanced economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,AE,United Arab Emirates,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,AL,Albania,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,AR,Argentina,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,AT,Austria,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
8,AU,Australia,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
9,BA,Bosnia and Herzegovina,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [23]:
metadata["CL_TC_BORROWERS"]

,Code,Description,URN
0,C,Non financial sector,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,G,General government,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,H,Households & NPISHs,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,N,Non-financial corporations,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,P,Private non-financial sector,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [24]:
metadata["CL_TC_LENDERS"]

,Code,Description,URN
0,A,All sectors,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,B,"Banks, domestic",urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [25]:
metadata["CL_VALUATION"]

,Code,Description,URN
0,M,Market value,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,N,Nominal value,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [26]:
metadata["CL_BIS_UNIT"].head(20)

,Code,Description,URN
0,000,Unknown,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,001,100 - yield,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,002,EUR / MWh,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,003,"Index, 1996 Jan 2 = 100",urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,004,Euro / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,005,US Dollar / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,006,US Dollar / lb,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,007,US Dollar / US gal,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
8,008,US Dollar / Million British Thermal Unit,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
9,009,US Dollar / Bushel,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [27]:
summary = []

for cl in dsd_response.codelist.values():

    summary.append({
        "Codelist ID": cl.id,
        "Name": str(cl.name),
        "Codes": len(cl.items)
    })

df_summary = pd.DataFrame(summary)

df_summary

,Codelist ID,Name,Codes
0,CL_ADJUST,Adjustment codelist,4
1,CL_AREA,Reference area code list,101
2,CL_BIS_UNIT,BIS_Unit,1096
3,CL_COLLECTION,Collection,10
4,CL_CONF_STATUS,Observation confidentiality code list,5
5,CL_DECIMALS,"Decimals codelist (BIS, ECB)",17
6,CL_FREQ,Code list for Frequency (FREQ),8
7,CL_OBS_STATUS,"Observation status codelist (BIS, ECB, Eurosta...",16
8,CL_SUB_CHANNEL,Submission channel,3
9,P,,1


In [39]:
metadata["CL_AREA"].query(
    "Description.str.contains('India', case=False)",
    engine="python"
)

,Code,Description,URN
47,IN,India,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [40]:
metadata["CL_BIS_UNIT"].query(
    "Description.str.contains('US dollar', case=False)",
    engine="python"
)

,Code,Description,URN
5,005,US Dollar / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,006,US Dollar / lb,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,007,US Dollar / US gal,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
8,008,US Dollar / Million British Thermal Unit,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
9,009,US Dollar / Bushel,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
10,010,US Dollar / Bitcoin,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
14,103,Chained 1992 US Dollar,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
19,108,Chained 1996 US Dollar,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
29,118,Constant 1972 US Dollar,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
39,128,Constant 1982 US Dollar,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [28]:
metadata["CL_AREA"].query(
    "Description.str.contains('Switzerland', case=False)",
    engine="python"
)

,Code,Description,URN
20,CH,Switzerland,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [29]:
metadata["CL_BIS_UNIT"].query(
    "Description.str.contains('Percent', case=False)",
    engine="python"
)

,Code,Description,URN
280,369,Percentage change against January 1987,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
281,370,Percentage Points,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
684,770,Percentage of GDP,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
713,799,Percentage of GDP (using PPP exchange rates),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [30]:
# ----------------------------------------------------
# Explore a Codelist
# ----------------------------------------------------

def explore_codelist(codelist_id):
    """
    Display all codes belonging to a BIS SDMX Codelist.
    """

    if codelist_id not in metadata:
        raise ValueError(f"{codelist_id} not found.")

    return metadata[codelist_id]

In [31]:
explore_codelist("CL_FREQ")

,Code,Description,URN
0,A,Annual,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,B,Daily - business week (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,D,Daily,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,E,Event (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,H,Half-yearly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,M,Monthly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,Q,Quarterly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,W,Weekly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [32]:
explore_codelist("CL_AREA")

,Code,Description,URN
0,1X,ECB,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,4T,Emerging market economies (aggregate),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,5A,All reporting economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,5R,Advanced economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,AE,United Arab Emirates,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
...,...,...,...
96,VN,Vietnam,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
97,XM,Euro area,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
98,XW,World,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
99,ZA,South Africa,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [33]:
explore_codelist("CL_TC_BORROWERS")

,Code,Description,URN
0,C,Non financial sector,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,G,General government,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,H,Households & NPISHs,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,N,Non-financial corporations,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,P,Private non-financial sector,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [34]:
explore_codelist("CL_BIS_UNIT").head(30)

,Code,Description,URN
0,000,Unknown,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,001,100 - yield,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,002,EUR / MWh,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,003,"Index, 1996 Jan 2 = 100",urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,004,Euro / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,005,US Dollar / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,006,US Dollar / lb,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,007,US Dollar / US gal,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
8,008,US Dollar / Million British Thermal Unit,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
9,009,US Dollar / Bushel,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [35]:
# ----------------------------------------------------
# Explore a Codelist (Enhanced)
# ----------------------------------------------------

def explore_codelist(codelist_id):
    """
    Explore a BIS SDMX Codelist.
    """

    if codelist_id not in dsd_response.codelist:
        raise ValueError(f"{codelist_id} not found.")

    cl = dsd_response.codelist[codelist_id]

    print("=" * 70)
    print(f"Codelist ID : {cl.id}")
    print(f"Name        : {cl.name}")
    print(f"Maintainer  : {cl.maintainer}")
    print(f"Version     : {cl.version}")
    print(f"Codes       : {len(cl.items)}")
    print("=" * 70)

    return metadata[codelist_id]

In [36]:
explore_codelist("CL_FREQ")

Codelist ID : CL_FREQ
Name        : Code list for Frequency (FREQ)
Maintainer  : BIS
Version     : 1.0
Codes       : 8


,Code,Description,URN
0,A,Annual,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,B,Daily - business week (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,D,Daily,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,E,Event (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,H,Half-yearly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,M,Monthly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,Q,Quarterly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,W,Weekly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [37]:
# ----------------------------------------------------
# Build a Codelist Catalog
# ----------------------------------------------------

catalog = []

for cl in dsd_response.codelist.values():

    catalog.append({
        "Codelist ID": cl.id,
        "Name": str(cl.name),
        "No. of Codes": len(cl.items)
    })

df_catalog = (
    pd.DataFrame(catalog)
      .sort_values("Codelist ID")
      .reset_index(drop=True)
)

df_catalog

,Codelist ID,Name,No. of Codes
0,CL_ADJUST,Adjustment codelist,4
1,CL_AREA,Reference area code list,101
2,CL_BIS_UNIT,BIS_Unit,1096
3,CL_COLLECTION,Collection,10
4,CL_CONF_STATUS,Observation confidentiality code list,5
5,CL_DECIMALS,"Decimals codelist (BIS, ECB)",17
6,CL_FREQ,Code list for Frequency (FREQ),8
7,CL_OBS_STATUS,"Observation status codelist (BIS, ECB, Eurosta...",16
8,CL_SUB_CHANNEL,Submission channel,3
9,CL_TC_BORROWERS,Borrowers,5


In [42]:
type(dsd_response)

sdmx.message.StructureMessage

In [43]:
# ----------------------------------------------------
# Extract the Data Structure Definition (DSD)
# ----------------------------------------------------

dsd = next(iter(dsd_response.structure.values()))

print(dsd)

BIS_TOTAL_CREDIT


In [44]:
# ----------------------------------------------------
# Dimension → Codelist Mapping
# ----------------------------------------------------

mapping = []

for dim in dsd.dimensions:

    rep = dim.local_representation

    codelist = None

    if rep is not None and rep.enumerated is not None:
        codelist = rep.enumerated.id

    mapping.append({
        "Dimension": dim.id,
        "Dimension Name": str(dim.concept_identity.name),
        "Codelist": codelist
    })

df_mapping = pd.DataFrame(mapping)

df_mapping

,Dimension,Dimension Name,Codelist
0,FREQ,Frequency,CL_FREQ
1,BORROWERS_CTY,Borrowers' country,CL_AREA
2,TC_BORROWERS,Borrowing sector,CL_TC_BORROWERS
3,TC_LENDERS,Lending sector,CL_TC_LENDERS
4,VALUATION,Valuation method,CL_VALUATION
5,UNIT_TYPE,Unit type,CL_BIS_UNIT
6,TC_ADJUST,Adjustment,CL_ADJUST
7,TIME_PERIOD,Time period or range,None


# Instead of requiring users to remember codelist IDs, let's make the explorer dimension-centric.

# ----------------------------------------------------

In [45]:
# ----------------------------------------------------
# Explore a Dimension
# ----------------------------------------------------

def explore_dimension(dimension_id):
    """
    Display all valid values for a given Dimension.
    """

    # Find the matching dimension
    match = df_mapping[df_mapping["Dimension"] == dimension_id]

    if match.empty:
        raise ValueError(f"Dimension '{dimension_id}' not found.")

    codelist_id = match.iloc[0]["Codelist"]

    if pd.isna(codelist_id):
        print(f"Dimension '{dimension_id}' has no associated codelist.")
        return

    print("=" * 70)
    print(f"Dimension : {dimension_id}")
    print(f"Name      : {match.iloc[0]['Dimension Name']}")
    print(f"Codelist  : {codelist_id}")
    print("=" * 70)

    return metadata[codelist_id]

In [46]:
explore_dimension("FREQ")

Dimension : FREQ
Name      : Frequency
Codelist  : CL_FREQ


,Code,Description,URN
0,A,Annual,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,B,Daily - business week (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,D,Daily,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,E,Event (not supported),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,H,Half-yearly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
5,M,Monthly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
6,Q,Quarterly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
7,W,Weekly,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [47]:
explore_dimension("BORROWERS_CTY")

Dimension : BORROWERS_CTY
Name      : Borrowers' country
Codelist  : CL_AREA


,Code,Description,URN
0,1X,ECB,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,4T,Emerging market economies (aggregate),urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,5A,All reporting economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,5R,Advanced economies,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,AE,United Arab Emirates,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
...,...,...,...
96,VN,Vietnam,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
97,XM,Euro area,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
98,XW,World,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
99,ZA,South Africa,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


In [48]:
explore_dimension("UNIT_TYPE")

Dimension : UNIT_TYPE
Name      : Unit type
Codelist  : CL_BIS_UNIT


,Code,Description,URN
0,000,Unknown,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1,001,100 - yield,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
2,002,EUR / MWh,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
3,003,"Index, 1996 Jan 2 = 100",urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
4,004,Euro / troy ounce,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
...,...,...,...
1091,ZMK,Zambian Kwacha,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1092,ZMW,Zambian Kwacha,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1093,ZWD,Zimbabwe Dollar,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...
1094,ZWG,Zimbabwe Gold,urn:sdmx:org.sdmx.infomodel.codelist.Code=BIS:...


# Summary

In this notebook, we explored the **structural metadata** returned by the BIS SDMX API and built a simple **Metadata Explorer**.

Starting with a **StructureMessage** (or dsd_response - This response is a StructureMessage, which contains not only the DSD but also other structural metadata such as Codelists, Concept Schemes, and Organisation Schemes.), we discovered that it contains not only the **Data Structure Definition (DSD)** but also several other metadata collections, including **Codelists**, **Concept Schemes**, and **Organisation Schemes**.

Our primary focus was on **Codelists**, as they define the **valid values** that can be used for many dimensions in an SDMX dataset.

Specifically, we:

- Explored the **StructureMessage** returned by the BIS API.
- Identified all **Codelists** associated with the selected Dataflow.
- Examined the structure of **Codelist** and **Code** objects.
- Built summary tables describing the available Codelists.
- Converted Codelists into pandas DataFrames for easy analysis.
- Created a reusable **Metadata Explorer** to inspect any Codelist.
- Mapped **Dimensions** to their corresponding **Codelists**.
- Built helper functions to explore valid values for both Codelists and Dimensions.

By the end of this notebook, we can now answer questions such as:

- Which countries are available?
- Which reporting frequencies are supported?
- Which borrower and lender classifications exist?
- Which units of measurement are available?
- What are the valid values for any queryable dimension?

Importantly, all of this metadata was discovered **without downloading a single observation**.

---

## Progress So Far

Our SDMX discovery journey now looks like this:

```text
Notebook 1
Discover Dataflows
        │
        ▼
Notebook 2
Explore the Data Structure Definition (DSD)
        │
        ▼
Notebook 3
Build a Metadata Explorer
        │
        ▼
Valid Codes for Every Dimension
```

---

## What Next?

We now understand:

- **What datasets are available** (Dataflows)
- **How a dataset is structured** (DSD)
- **Which values are valid for each dimension** (Codelists)

In the next notebook, we will use this metadata to **construct valid SDMX query keys** and learn how to retrieve observations from the BIS SDMX API.